# Screen random initialisations for cyclic behaviour

This lightweight diagnostic runs the Pre Norm model with projected input injection
from `initialisation.ipynb` for ten fixed seeds. Each panel compares the output of
the first recurrent layer at every recurrence with that layer's final observed state.
Periodic returns toward cosine similarity one are evidence of an orbit worth studying.

The default evaluates only the first causal token, which is mathematically independent
of later tokens and makes this candidate scan much cheaper. Set `sequence_length = 32`
to verify a selected seed with the token average used by `initialisation.ipynb`.


In [ ]:
seeds = list(range(42, 52))
recurrences = 128
layers = 12
width = 512
num_heads = 4
sequence_length = 1  # Use 32 to reproduce the main notebook's token average.
vocab_size = 32000
device = "auto"
cache_dir = None
cache_mode = None  # reuse, require, or off; defaults from the environment.
analysis_version = 1

In [ ]:
import gc
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dataclasses import asdict

from experiments.cache import ExperimentCache
from looped_internals.models import MinimalRecurrentConfig, MinimalRecurrentLM

plt.style.use(ROOT / "plots" / "matlab.mplstyle")
resolved_device = "cuda" if device == "auto" and torch.cuda.is_available() else device
if resolved_device == "auto":
    resolved_device = "cpu"

result_cache = ExperimentCache(cache_dir, cache_mode)

In [ ]:
model_config = MinimalRecurrentConfig(
    d_model=width,
    n_heads=num_heads,
    n_layers=layers,
    num_cycles=recurrences,
    d_vocab=vocab_size,
    input_injection="projection",
    norm_pre_attn=True,
    norm_post_attn=False,
    norm_pre_mlp=True,
    norm_post_mlp=False,
    norm_post_attn_residual=False,
    norm_post_mlp_residual=False,
    norm_post_cycle=False,
)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@torch.inference_mode()
def final_state_cosine_curve(seed):
    """Return token-averaged first-layer similarities to its final state."""
    set_seed(seed)
    model = MinimalRecurrentLM(model_config).to(resolved_device).eval()
    token_ids = torch.randint(
        vocab_size,
        (1, sequence_length),
        generator=torch.Generator().manual_seed(seed),
    ).to(resolved_device)

    states = []

    def capture_first_layer(_module, _inputs, output):
        states.append(output[0].detach().float().cpu())

    handle = model.transformer_layers[0].register_forward_hook(capture_first_layer)
    try:
        model.forward_no_logits(model.embed_tokens(token_ids))
    finally:
        handle.remove()

    states = torch.stack(states)
    if len(states) != recurrences:
        raise RuntimeError(f"Captured {len(states)} states, expected {recurrences}")
    per_token = F.cosine_similarity(states, states[-1].expand_as(states), dim=-1)
    curve = per_token.mean(-1)

    del model, token_ids
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return curve

## Generate only the required trajectories

Each seed creates one model and, by default, one single-token rollout. The hook retains
only the first physical layer's states; logits, attention patterns, other layer outputs,
pairwise comparisons, Jacobians, and distance metrics are never computed.


In [ ]:
curves = {
    run_seed: result_cache.run(
        "initialisation_seed_screen",
        result_cache.key(
            analysis_version=analysis_version,
            model_config=asdict(model_config),
            seed=run_seed,
            sequence_length=sequence_length,
            device=resolved_device,
        ),
        final_state_cosine_curve,
        run_seed,
    )
    for run_seed in tqdm(seeds, desc="Initialisation seeds")
}

## Same-layer final-state cosine similarity

All panels share the same limits. Look for repeated, regularly spaced peaks rather than
a single monotonic approach to the final reference; the seed is printed above each curve.


In [ ]:
ncols = min(5, len(seeds))
nrows = int(np.ceil(len(seeds) / ncols))
fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(2.4 * ncols, 2.35 * nrows),
    sharex=True,
    sharey=True,
    layout="constrained",
    squeeze=False,
)
x = np.arange(1, recurrences + 1)
minimum = min(curve.min().item() for curve in curves.values())
padding = max(0.02, 0.05 * (1 - minimum))

for ax, (run_seed, curve) in zip(axes.flat, curves.items(), strict=False):
    ax.plot(x, curve.numpy(), linewidth=1)
    ax.axhline(1, color="0.7", linewidth=0.6, linestyle="--")
    ax.set_title(f"Seed {run_seed}")
    ax.set_ylim(max(-1.01, minimum - padding), 1.01)
for ax in axes.flat[len(seeds) :]:
    ax.set_visible(False)

fig.supxlabel("Recurrence")
fig.supylabel("Final-state cosine similarity (same layer)")
plt.show()